# Threshold curve in the $(p, p_{\mathrm{bell},z})$ plane — Bell models & equivalents

Headline figure: every fitted threshold is a 2D point $(p^*, p_{\mathrm{bell},z}^*)$
on the noise plane. Each noise model is drawn as a separate curve.

- **bias scan**: direction $\beta = p_{\mathrm{bell},z}/p$, threshold lands at
  $(p^*, \beta \cdot p^*)$.
- **fixed-$p_{\mathrm{bell},z}$ scan**: $p_{\mathrm{bell},z}$ held constant,
  threshold lands at $(p^*, p_{\mathrm{bell},z}^{\mathrm{fixed}})$.

`biased_bell_v2` / `biased_bell_v3` fits come pre-computed from
`threshold_comparison_bell.ipynb` (`out/collection/threshold_bell_fits.csv`).
The single-ancilla equivalents — `biased_bell_v3_equivalent` and
`biased_bell_v3_equivalent_majority` — are fitted in-place from their raw decode
CSVs (`out/collection/threshold_equivalent,*.csv` and
`threshold_equivalent_majority,*.csv`, written by `tools/equivalent/run_decode*.sh`).
Each equivalent should land on top of `biased_bell_v3`; the majority variant
trades extra rounds for a lower effective measurement error.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Shared analysis backend (also used by tools/plot_thresholds_bell.py) so the
# notebook and the CLI produce identical figures.
from bell_threshold_analysis import (
    load_decode_csvs,
    fit_thresholds,
    plot_threshold_plane,
    plot_threshold_vs_fixed_pbz,
    plot_combined_plane,
)

%load_ext autoreload
%autoreload 2

In [2]:
fits_path = Path('../out/collection/threshold_bell_fits.csv')
if not fits_path.exists():
    raise FileNotFoundError(
        f'No fits at {fits_path}. Run threshold_comparison_bell.ipynb first.'
    )

fits_df = pd.read_csv(fits_path)

# Backwards-compat: older fits CSVs lacked the noise_model column.
if 'noise_model' not in fits_df.columns:
    fits_df['noise_model'] = 'biased_bell_v2'

print('fits per noise_model:')
print(fits_df.groupby('noise_model').size())
fits_df

fits per noise_model:
noise_model
biased_bell_v2    12
biased_bell_v3    11
dtype: int64


,noise_model,direction,direction_kind,bias,p_bell_z_fixed,p_th,p_th_err,mu,mu_err,p_init,n_points
0,biased_bell_v2,bias=0,bias,0.0,NaN,0.002418,2.159251e-05,0.910499,0.044097,0.002512,15
1,biased_bell_v2,bias=1,bias,1.0,NaN,0.002047,1.233021e-05,0.971927,0.035426,0.002154,14
2,biased_bell_v2,bias=2,bias,2.0,NaN,0.001701,2.051408e-05,0.756509,0.026000,0.001585,13
3,biased_bell_v2,bias=3,bias,3.0,NaN,0.001486,9.182376e-06,0.954573,0.026916,0.001585,14
4,biased_bell_v2,bias=5,bias,5.0,NaN,0.001196,1.259919e-05,0.815859,0.025822,0.001166,13
5,biased_bell_v2,bias=10,bias,10.0,NaN,0.000795,4.923095e-06,0.880964,0.021115,0.000858,14
6,biased_bell_v2,bias=20,bias,20.0,NaN,0.000492,6.146875e-06,0.713299,0.025259,0.000464,13
7,biased_bell_v2,bias=30,bias,30.0,NaN,0.000348,2.581181e-06,0.791440,0.017868,0.000341,13
8,biased_bell_v2,bias=50,bias,50.0,NaN,0.000220,1.991365e-06,0.884524,0.030904,0.000251,12
9,biased_bell_v2,bias=100,bias,100.0,NaN,0.000116,8.823396e-07,0.963759,0.036348,0.000136,9


In [ ]:
# Append the equivalent models. Unlike v2/v3 (pre-fitted in
# threshold_comparison_bell.ipynb), the equivalent results are raw decode CSVs
# from tools/equivalent/run_decode*.sh, so we fit them here with the same shared
# locate -> quadratic-FSS pipeline (src/bell_threshold_analysis.py), then concat
# onto fits_df. Each CSV's metadata carries the right noise_model, so the
# single-ancilla equivalent and its majority-vote variant separate automatically.
EQUIV_CSV_GLOBS = [
    '../out/collection/threshold_equivalent,*.csv',
    '../out/collection/threshold_equivalent_majority,*.csv',
]
# Equivalent model names this cell owns (stale rows are dropped before append).
EQUIV_MODELS = [
    'biased_bell_v3_equivalent',
    'biased_bell_v3_equivalent_majority',
]

try:
    edf = load_decode_csvs(EQUIV_CSV_GLOBS)
except FileNotFoundError:
    print(f'No equivalent CSVs matched {EQUIV_CSV_GLOBS}; skipping.')
    equiv_fits_df = pd.DataFrame()
else:
    # LOGICAL_CUTOFF=0.45 here (looser than v2/v3) to keep more points.
    equiv_fits_df = fit_thresholds(edf, n_per_side=2, logical_cutoff=0.45, verbose=True)
    # Drop any stale equivalent rows, then append the freshly-fitted ones.
    fits_df = fits_df[~fits_df['noise_model'].isin(EQUIV_MODELS)]
    fits_df = pd.concat([fits_df, equiv_fits_df], ignore_index=True)

    print(f'\nFitted {len(equiv_fits_df)} equivalent-model directions.')
    print('fits per noise_model (after append):')
    print(fits_df.groupby('noise_model').size())

equiv_fits_df if not equiv_fits_df.empty else fits_df

In [ ]:
# Summary of what will be plotted. The per-model bias / fixed-pbz grouping now
# lives inside the plot functions (bell_threshold_analysis), so all the plotting
# cells below need is `fits_df`.
print('fits per (noise_model, direction_kind):')
print(fits_df.groupby(['noise_model', 'direction_kind']).size())

In [ ]:
# Each fitted threshold -> point (p_th, beta * p_th) on the (p, p_bell_z) plane.
# One curve per noise model. v3, its single-ancilla equivalent, and the
# majority-vote variant should fall on top of each other (equivalence check).
# Same figure tools/plot_thresholds_bell.py writes as threshold_bell_curves.pdf.
fig = plot_threshold_plane(fits_df)
Path('../out/figures').mkdir(parents=True, exist_ok=True)
fig.savefig('../out/figures/threshold_bell_curves.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# p_th vs fixed p_bell_z (independent scan, easier to interpret experimentally
# because p_bell_z is set by hardware). One curve per noise model.
fig = plot_threshold_vs_fixed_pbz(fits_df)
if fig is not None:
    plt.show()
else:
    print('No fixed-pbz fits to plot.')

In [ ]:
# Combined view: both scans on one (p_bell_z, p) plane, one set of curves per
# noise model.
# - bias scan: each point is (p_th * beta, p_th).
# - fixed-pbz scan: each point is (p_bell_z_fixed, p_th).
fig = plot_combined_plane(fits_df)
plt.show()